# Market Making with Alpha - APT

## Overview

Continuing from [Market Making with Alpha - Basis](https://hftbacktest.readthedocs.io/en/latest/tutorials/Market%20Making%20with%20Alpha%20-%20Basis.html), this example demonstrates market making based on [Arbitrage Pricing Theory](https://en.wikipedia.org/wiki/Arbitrage_pricing_theory).



<div class="alert alert-info">
    
**Note:** This example is for educational purposes only and demonstrates effective strategies for high-frequency market-making schemes. All backtests are based on a 0.005% rebate, the highest market maker rebate available on Binance Futures. See <a href="https://www.binance.com/en/support/announcement/binance-updates-usd%E2%93%A2-margined-futures-liquidity-provider-program-2024-06-03-fefc6aa25e0947e2bf745c1c56bea13e">Binance Upgrades USDⓢ-Margined Futures Liquidity Provider Program</a> for more details.
    
</div>

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `Market Making with Alpha - APT.ipynb` (`market_making_alpha_apt`).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context('0804T004')
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('market_making_alpha_apt', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

Under Arbitrage Pricing Theory, the relationship between futures return and spot return can be expressed as:

$Return_{futures} = \alpha + \beta_{spot} * Return_{spot}$

Under the assumption that $\beta_{spot}$ = 1 and $\alpha$ = 0, the futures return should be equal to the spot return. This also implies that any residual movement is mean-reverting to zero, similar to what is shown in the basis example.

**Extending the Model**

Beyond this basic relationship, additional return-contributing factors can be incorporated. For instance, returns from other exchanges’ Bitcoin markets, such as:

* CME Bitcoin futures, Bybit's BTC futures and other platforms's BTC futures
* Bitcoin ETFs
* Spot prices from Coinbase, Kraken, and other platforms

Moreover, this is not limited to the same asset. Other cryptocurrencies, traditional assets, and macroeconomic indices can be considered, such as:

* Ethereum (ETH)
* S&P 500
* Dollar Index

Additionally, market microstructure factors, such as order book imbalance, can further enhance the model, as demonstrated in [our other example](https://hftbacktest.readthedocs.io/en/latest/tutorials/Market%20Making%20with%20Alpha%20-%20Order%20Book%20Imbalance.html).

This broader framework allows for a more comprehensive understanding of price movements and their underlying drivers.

As demonstrated in the basis example, BTCFDUSD behaves in a similar manner.

## Integrating Grid Trading

## Extension to the Multi-Factor Model

### Simple Form: Utilizing Both BTCUSDT and BTCFDUSD Spot Returns
One of the simplest ways to incorporate both BTCUSDT and BTCFDUSD spot returns is by using equal beta, which takes the average of these two values.

### MLR: Utilizing Both BTCUSDT and BTCFDUSD Spot Returns
Since these two variables are highly correlated, proper handling is necessary. One approach is the residual method, but other techniques, such as PCA, can also be used to eliminate correlation. Additionally, when applying Multiple Linear Regression, you may need to constrain beta values within a specific range, such as ensuring they remain positive. In such cases, more advanced techniques can be utilized.

Given the assumption that the deviation in futures returns mean-reverts to the spot return, the target is set as the spot return.

## A Comprehensive Framework for Pricing Models
Let's explore a more generalized approach to asset pricing from a conceptual standpoint.

### Core Market Drivers (Primary Price Movement)
The first component represents price movement driven by core market instruments — typically spot and futures markets across major venues. It can be expressed as:

```
Return_BTC = β00 * BTCUSDT_spot
           + β01 * BTCFDUSD_spot
           + β02 * BTCUSDT_futures_on_exchange1
           + β03 * BTCUSD_inverse_perpetual1
           + β04 * BTCUSD_CME_futures
           + β05 * BTC_ETF1
           + ...
```
This model doesn’t require all components. You can identify the most influential inputs using statistical methods (e.g., regression, PCA, or Granger causality), or by analyzing market depth, trading volume, and lead-lag relationships between exchanges. You can also see the importance of considering the Trad-Fi market — including CME futures, Bitcoin ETFs, and equity markets — by comparing weekday returns, which highlight a different return profile on weekends (typically better).

### Broader Market Influence (Cross-Asset Correlation)
Bitcoin’s price can also be influenced by the movement of other major cryptocurrencies — similar to how components interact with an index:

```
Return_CrossAsset = β10 * ETHUSDT_futures
                  + β11 * SOLUSDT_futures
                  + ...
```
You may also use spot markets, but it’s important to select markets with high liquidity and trading volume, as they are more likely to drive broader price movements.

### Microstructure Signals & Alpha Factors
Short-term price forecasts can benefit from market microstructure data and proprietary alpha signals. These might include:

```
Return_Alpha = β20 * OrderBookImbalance1
             + β21 * OrderBookImbalance2
             + β22 * FundingRateAlpha
             + β23 * OpenInterestAlpha
             + ...
             + β2n * CustomAlpha_n
```
These signals are especially valuable for short-horizon trading, such as high-frequency or latency-sensitive strategies.

### Combined Pricing Model
All components can be integrated into a single predictive return model:

```
Forecast_Return = β0 * Return_Self
                + Return_BTC
                + Return_CrossAsset
                + Return_Alpha
```
Return_BTC and Return_CrossAsset reflect structural or market-level influences and Return_Alpha represents short-term, predictive signals based on microstructure or custom models. In addition, defining fair value price is crucial, as it shapes your trading setup. A straight defintion is to forecast future returns (e.g., 10s, 30s, 1min, 5min), depending on the trading horizon. The regression target should then be this fair value price.

### Exchange-Specific Application
The effectiveness of this model may depend on your forecasting horizon:

For medium-term forecasts (e.g., 1–5 minutes), this model can generalize across major venues such as Binance, Bybit, OKX, and Hyperliquid.

For very short-term trading (e.g., sub-second to a few seconds), you need to account for exchange-specific dynamics such as latency, liquidity, and order flow patterns.

For example, since Binance Futures has the highest trading volume, its price movements often lead the market. Other exchanges may lag behind.

The simplest cross-exchange model might look like this:

```
Return_Bybit ≈ Return_Binance
```

This setup is useful for cross-exchange arbitrage or liquidity-driven strategies, where you exploit short-term dislocations between platforms.

More examples incorporating additional factors beyond BTC returns and cross-exchange cases such as described in https://hangukquant.github.io/scripts/market_making, will be added.